## Colab setup

In Colab, either upload the CSV files into the runtime or clone your GitHub repository first.

## How to upload the dataset into Google Colab

You can use any one of these simple methods.

### Option 1: Upload files directly
Use this when the dataset is small enough.
1. Open the folder icon in Colab.
2. Click Upload.
3. Select all CSV files.
4. They will appear in `/content`.
5. Keep `ROOT_DIR = Path('/content')`.

### Option 2: Mount Google Drive
Use this when you want to keep the dataset in Drive.
```python
from google.colab import drive
drive.mount('/content/drive')
```
Then set `ROOT_DIR` to the folder that contains the CSV files, for example:
```python
ROOT_DIR = Path('/content/drive/MyDrive/project2')
```

### Option 3: Clone your GitHub repository
Use this when your dataset and notebooks are already in GitHub.
```python
!git clone https://github.com/your-username/your-repo.git
ROOT_DIR = Path('/content/your-repo')
```

For this assignment, Option 2 or Option 3 is usually the cleanest choice.

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

ROOT_DIR = Path('/content')
OUTPUT_DIR = ROOT_DIR / 'outputs_python'
OUTPUT_DIR.mkdir(exist_ok=True)

print('Set ROOT_DIR to the folder that contains your CSV files.')

In [ ]:
ZONE_MAP = {
    'airport': 'Airport',
    'central': 'Central',
    'ctr': 'Central',
    'east': 'East',
    'north': 'North',
    'south': 'South',
    'west': 'West',
    'riverside': 'Riverside',
}

DATE_COLUMNS = {
    'customers': ['signup_date'],
    'orders': ['order_created_at'],
    'deliveries': ['dispatch_time', 'delivery_completed_at'],
    'incidents': ['reported_at'],
    'complaints': ['created_at'],
    'app_events': ['event_timestamp'],
    'vehicles': ['commission_date'],
}

def clean_zone(value):
    if pd.isna(value):
        return pd.NA
    text = str(value).strip().lower().replace(' ', '')
    return ZONE_MAP.get(text, str(value).strip().title())

def load_table(name):
    df = pd.read_csv(ROOT_DIR / f'{name}.csv')
    for col in DATE_COLUMNS.get(name, []):
        df[col] = pd.to_datetime(df[col], errors='coerce')
    return df

In [ ]:
tables = {name: load_table(name) for name in [
    'customers', 'drivers', 'vehicles', 'hubs', 'orders',
    'deliveries', 'incidents', 'complaints', 'app_events'
]}

tables['customers']['home_zone_clean'] = tables['customers']['home_zone'].apply(clean_zone)
tables['drivers']['base_zone_clean'] = tables['drivers']['base_zone'].apply(clean_zone)
tables['vehicles']['assigned_zone_clean'] = tables['vehicles']['assigned_zone'].apply(clean_zone)
tables['hubs']['hub_zone_clean'] = tables['hubs']['zone'].apply(clean_zone)
tables['orders']['pickup_zone_clean'] = tables['orders']['pickup_zone'].apply(clean_zone)
tables['orders']['dropoff_zone_clean'] = tables['orders']['dropoff_zone'].apply(clean_zone)
tables['app_events']['zone_context_clean'] = tables['app_events']['zone_context'].apply(clean_zone)

for name, df in tables.items():
    print(name, df.shape)

In [ ]:
quality_rows = []
for name, df in tables.items():
    total_cells = len(df) * len(df.columns)
    missing_cells = int(df.isna().sum().sum())
    quality_rows.append({
        'table_name': name,
        'row_count': len(df),
        'column_count': len(df.columns),
        'missing_cells': missing_cells,
        'missing_share': round(missing_cells / total_cells, 4) if total_cells else 0,
        'duplicate_rows': int(df.duplicated().sum())
    })

data_quality_summary = pd.DataFrame(quality_rows).sort_values('missing_share', ascending=False)
data_quality_summary

### Interpretation of the data quality check

This table shows that the joined analytical view will contain much more missing data than the original source tables. That is normal because not every order has a complaint, incident, or app event linked to it.

A good report point is that missing values here do not always mean bad data. In many cases, they simply mean no complaint was raised, no incident was recorded, or no matching event existed.

In [ ]:
complaints_summary = tables['complaints'].groupby('order_id', dropna=False).agg(
    complaint_count=('complaint_id', 'count'),
    complaint_compensation_total=('compensation_amount', 'sum')
).reset_index()

incidents_summary = tables['incidents'].groupby('delivery_id', dropna=False).agg(
    incident_count=('incident_id', 'count')
).reset_index()

app_event_summary = tables['app_events'].dropna(subset=['order_id']).groupby('order_id', dropna=False).agg(
    app_event_count=('event_id', 'count'),
    avg_api_latency_ms=('api_latency_ms', 'mean'),
    failed_app_events=('success_flag', lambda values: (values == 0).sum())
).reset_index()

order_delivery_view = (
    tables['orders']
    .merge(tables['customers'], on='customer_id', how='left')
    .merge(tables['deliveries'], on='order_id', how='left')
    .merge(tables['drivers'], on='driver_id', how='left')
    .merge(tables['vehicles'], on='vehicle_id', how='left')
    .merge(tables['hubs'], on='hub_id', how='left')
    .merge(complaints_summary, on='order_id', how='left')
    .merge(app_event_summary, on='order_id', how='left')
    .merge(incidents_summary, on='delivery_id', how='left')
)

order_delivery_view['delivery_hours'] = (
    order_delivery_view['delivery_completed_at'] - order_delivery_view['dispatch_time']
).dt.total_seconds() / 3600

order_delivery_view['was_late_or_failed'] = order_delivery_view['delivery_status'].isin(['Delayed', 'Failed']).astype(int)
order_delivery_view['has_complaint'] = order_delivery_view['complaint_count'].fillna(0).gt(0).astype(int)
order_delivery_view['has_incident'] = order_delivery_view['incident_count'].fillna(0).gt(0).astype(int)
order_delivery_view['manual_override_flag'] = order_delivery_view['manual_route_override_count'].fillna(0).gt(0).astype(int)
order_delivery_view['needs_attention'] = ((order_delivery_view['was_late_or_failed'] == 1) | (order_delivery_view['has_complaint'] == 1) | (order_delivery_view['has_incident'] == 1)).astype(int)

order_delivery_view.head()

In [ ]:
zone_summary = order_delivery_view.groupby('hub_zone_clean', dropna=False).agg(
    total_orders=('order_id', 'count'),
    delay_rate=('was_late_or_failed', 'mean'),
    complaint_rate=('has_complaint', 'mean'),
    avg_rating=('customer_rating_post_delivery', 'mean')
).reset_index().sort_values('delay_rate', ascending=False)

service_summary = order_delivery_view.groupby('service_type', dropna=False).agg(
    total_orders=('order_id', 'count'),
    complaint_rate=('has_complaint', 'mean'),
    avg_order_value=('order_value', 'mean')
).reset_index().sort_values('complaint_rate', ascending=False)

attention_orders = order_delivery_view[order_delivery_view['needs_attention'] == 1].copy()

zone_summary.head(), service_summary.head(), attention_orders.head()

In [ ]:
zone_chart = zone_summary.dropna(subset=['hub_zone_clean']).sort_values('delay_rate')

plt.figure(figsize=(8, 5))
plt.barh(zone_chart['hub_zone_clean'], zone_chart['delay_rate'], color='steelblue')
plt.title('Delay or fail rate by hub zone')
plt.xlabel('Delay or fail rate')
plt.ylabel('Hub zone')
plt.tight_layout()
plt.show()

## What to write after the code

Add 3 to 5 findings in simple business language, for example:
- which zone has the highest delay rate
- which service type has the highest complaint rate
- whether risky orders are concentrated in a few hubs